In [0]:
# ======================================
# Dataset: olist_order_payments
# Layer: Silver
# Source: Bronze (delta)
# Grain: order_id + payment_sequential
# ======================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
%run ../commons/silver_utils

In [0]:
%run ../../00_config/paths

In [0]:
df_bronze = spark.read.format('delta').load(BRONZE_ORDER_PAYMENTS_PATH)
df_bronze.printSchema()

In [0]:
df = normalize_columns(df_bronze)

In [0]:
df = df.select(
    F.col("order_id"),
    F.col("payment_sequential").cast(IntegerType()).alias("payment_sequential"),
    F.col("payment_installments").cast(IntegerType()).alias("payment_installments"),
    F.col("payment_value").cast(DecimalType(10, 2)).alias("payment_value"),
    F.trim(F.lower(F.col("payment_type"))).alias("payment_type")
)

In [0]:
df = df.filter(
    F.col("order_id").isNotNull() &
    F.col("payment_sequential").isNotNull()
)


In [0]:
df = df.filter(
    (F.col("payment_value") > 0) &
    (F.col("payment_installments") >= 1)
)


In [0]:
VALID_PAYMENT_TYPES = [
    "credit_card", "boleto", "voucher",
    "debit_card", "not_defined"
]

df = df.filter(F.col("payment_type").isin(VALID_PAYMENT_TYPES))


In [0]:
df.printSchema()

In [0]:
print("Bronze count:", df_bronze.count())
print("Silver count:", df.count())


In [0]:
write_silver(df, SILVER_ORDER_PAYMENTS_PATH)